In [1]:
import time

# We will need Qibo to write our quantum circuit
from qibo import Circuit, gates, set_backend

# We will need mpstab to construct a surrogate
from mpstab import HSMPO

In [2]:
set_backend("qibojit", platform="numba")

[Qibo 0.2.23|INFO|2026-02-23 13:55:01]: Using qibojit (numba) backend on /CPU:0


In [3]:
n = 10

circ = Circuit(nqubits=n)
circ.add(gates.H(0))
[circ.add(gates.CNOT(q, q+1)) for q in range(n - 1)]

circ.draw()

0: ─H─o─────────────────
1: ───X─o───────────────
2: ─────X─o─────────────
3: ───────X─o───────────
4: ─────────X─o─────────
5: ───────────X─o───────
6: ─────────────X─o─────
7: ───────────────X─o───
8: ─────────────────X─o─
9: ───────────────────X─


In [4]:
surry = HSMPO(circ)

In [5]:
# %time
surry.expectation(
    observable="Z"*n
)

/Users/robbiati/Documents/tac/lib/python3.12/site-packages/cotengra/hyperoptimizers/hyper.py:36: UserWarning: Couldn't import `kahypar` - skipping from default hyper optimizer and using basic `labels` method instead. `kahypar` is highly recommended for the best quality contraction paths.
  warnings.warn(


np.float64(1.0)

In [6]:
from mpstab.models.ansatze import HardwareEfficient 
from mpstab.utils import obs_string_to_qibo_hamiltonian

def execute_circuit(nqubits, nlayers):
    obs_str = "Z" * nqubits

    ans = HardwareEfficient(nqubits=nqubits, nlayers=nlayers)
    hs = HSMPO(ansatz=ans)

    it = time.time()
    expval_mpstab = hs.expectation(obs_str)
    time_mpstab = time.time() - it
    
    ham = obs_string_to_qibo_hamiltonian(obs_str)
    it = time.time()
    expval_qibo = ham.expectation_from_state(
        ans.circuit().state()
    )
    time_qibo = time.time() - it
    return (expval_mpstab, expval_qibo), (time_mpstab, time_qibo)

In [7]:
(expval_mpstab, expval_qibo), (time_mpstab, time_qibo) = execute_circuit(nqubits=12, nlayers=2)

print(f"MPSTAB expectation value: {expval_mpstab}, time taken: {time_mpstab:.4f} seconds")
print(f"Qibo expectation value: {expval_qibo}, time taken: {time_qibo:.4f} seconds")

/Users/robbiati/Documents/tac/lib/python3.12/site-packages/autoray/autoray.py:96: RuntimeWarning: divide by zero encountered in matmul
  return func(*args, **kwargs)
/Users/robbiati/Documents/tac/lib/python3.12/site-packages/autoray/autoray.py:96: RuntimeWarning: overflow encountered in matmul
  return func(*args, **kwargs)
/Users/robbiati/Documents/tac/lib/python3.12/site-packages/autoray/autoray.py:96: RuntimeWarning: invalid value encountered in matmul
  return func(*args, **kwargs)
/Users/robbiati/Documents/tac/lib/python3.12/site-packages/qibo/hamiltonians/hamiltonians.py:545: RuntimeWarning: divide by zero encountered in matmul
  result = reduce(
/Users/robbiati/Documents/tac/lib/python3.12/site-packages/qibo/hamiltonians/hamiltonians.py:545: RuntimeWarning: overflow encountered in matmul
  result = reduce(
/Users/robbiati/Documents/tac/lib/python3.12/site-packages/qibo/hamiltonians/hamiltonians.py:545: RuntimeWarning: invalid value encountered in matmul
  result = reduce(


MPSTAB expectation value: -0.0789404501798377, time taken: 0.4700 seconds
Qibo expectation value: -0.07894013378649811, time taken: 20.6114 seconds


/Users/robbiati/Documents/tac/lib/python3.12/site-packages/qibo/backends/numpy.py:796: RuntimeWarning: divide by zero encountered in matmul
  hstate = hamiltonian @ state
/Users/robbiati/Documents/tac/lib/python3.12/site-packages/qibo/backends/numpy.py:796: RuntimeWarning: overflow encountered in matmul
  hstate = hamiltonian @ state
/Users/robbiati/Documents/tac/lib/python3.12/site-packages/qibo/backends/numpy.py:796: RuntimeWarning: invalid value encountered in matmul
  hstate = hamiltonian @ state


In [8]:
from qibo.symbols import X, Y, Z
from qibo.hamiltonians import SymbolicHamiltonian

In [16]:
nqubits = 9
ham_form = 0
for i in range(8):
    ham_form += (1) ** i * (i + 1) * Z(i) * Z(i+1)  
h = SymbolicHamiltonian(ham_form)


In [17]:
print(h.simple_terms)

ans = HardwareEfficient(nqubits=nqubits, nlayers=2)
hs = HSMPO(ansatz=ans)

([(1+0j), (2+0j), (3+0j), (4+0j), (5+0j), (6+0j), (7+0j), (8+0j)], ['ZZ', 'ZZ', 'ZZ', 'ZZ', 'ZZ', 'ZZ', 'ZZ', 'ZZ'], [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 8)])


In [18]:
hs.expectation(observable=h)

/Users/robbiati/Documents/tac/lib/python3.12/site-packages/autoray/autoray.py:96: RuntimeWarning: divide by zero encountered in matmul
  return func(*args, **kwargs)
/Users/robbiati/Documents/tac/lib/python3.12/site-packages/autoray/autoray.py:96: RuntimeWarning: overflow encountered in matmul
  return func(*args, **kwargs)
/Users/robbiati/Documents/tac/lib/python3.12/site-packages/autoray/autoray.py:96: RuntimeWarning: invalid value encountered in matmul
  return func(*args, **kwargs)


3.1799895679010546

In [19]:
h.expectation_from_state(ans.circuit().state())

np.float64(3.179989567901073)